# Optimización Automática de Hiperparámetros con Optuna

Este cuaderno implementa la sintonía fina de hiperparámetros para el modelo **MTDE-Net** utilizando **Optuna**, una librería de optimización bayesiana de alto rendimiento.

El algoritmo aprende de las ejecuciones previas (a través de TPE - Tree-structured Parzen Estimator) y descarta de manera temprana pruebas poco prometedoras mediante **Poda (Pruning)** para optimizar el tiempo de GPU.

**Nota sobre la reproducibilidad:** En esta versión, se fuerza el reseteo de la semilla global (`set_seed(42)`) al inicio de cada prueba (trial) en la función objetivo. Esto garantiza que todos los modelos comiencen exactamente con los **mismos pesos iniciales**, permitiendo que las variaciones observadas en el rendimiento se deban exclusivamente a los hiperparámetros y no a la suerte de la inicialización aleatoria.

In [1]:
import sys
import random
from pathlib import Path

# Añadir el directorio raíz al path de Python para permitir importaciones correctas
sys.path.append(str(Path.cwd().parent))

import numpy as np
import optuna
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset

from src.models.mtde_net import MTDE_Net
from src.loaders.mtde_net_loader import MultimodalThermalDataset
from src.utils import SqrtScaledMSELoss, eval_mtde_net_metrics, build_subject_split_plan

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

d:\ulima\Ulima_archivos\UL-2026-1\Seminario1\experimentacion\flir_test\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1. Definición de la Función Objetivo de Optuna (Con Semilla Fija)

In [2]:
metadata_path = "../processed_data/metadata_train.csv"

# Instanciar primero los datasets para que filtren y reinicien sus índices
train_ds_opt = MultimodalThermalDataset(metadata_csv=metadata_path, is_train=True)
val_ds_opt = MultimodalThermalDataset(metadata_csv=metadata_path, is_train=False)
train_ds_opt.root = Path("../processed_data")
val_ds_opt.root = Path("../processed_data")

# Utilizar train_ds_opt.df (ya filtrado) para construir el plan de partición de 6 folds
# Tania se integra al entrenamiento/validación para mayor representatividad
plan_opt = build_subject_split_plan(
    train_ds_opt.df,
    n_splits=6,
    seed=42,
    test_subjects=None,
    reserve_incomplete_for_test=True,
)

def objective(trial):
    # resetear la semilla al inicio de cada trial para reproducibilidad de pesos iniciales
    set_seed(42)
    
    # 1. Espacio de Búsqueda ampliado y acotado razonablemente para prevenir overfitting
    lr = trial.suggest_float("lr", 5e-5, 2e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    time_scale = 30.0
    
    # 2. Evaluación por Validación Cruzada de 6 folds sobre el Train/Val set (Sin tocar el Test Set)
    fold_maes = []
    
    # Iteramos con índice para poder reportar progreso y habilitar poda (pruning) entre folds
    for fold_idx, fold in enumerate(plan_opt.folds):
        train_loader = DataLoader(
            Subset(train_ds_opt, fold.train_indices), 
            batch_size=batch_size, 
            shuffle=True,
            drop_last=True
        )
        val_loader = DataLoader(
            Subset(val_ds_opt, fold.val_indices), 
            batch_size=batch_size
        )
        
        model = MTDE_Net(dropout=dropout).to(device)
        crit = SqrtScaledMSELoss(scale=None)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        
        # Configurar Early Stopping para cada Fold en el trial
        epochs = 50
        patience = 8
        best_fold_mae = float("inf")
        no_improvement = 0
        
        for ep in range(epochs):
            model.train()
            for x_img, x_tab, y in train_loader:
                x_img, x_tab, y = x_img.to(device), x_tab.to(device), y.to(device)
                opt.zero_grad(set_to_none=True)
                loss = crit(model(x_img, x_tab), y)
                loss.backward()
                opt.step()
                
            model.eval()
            val_m = eval_mtde_net_metrics(model, val_loader, device, scale=time_scale)
            v_mae = val_m["mae"]
            if v_mae < best_fold_mae - 0.5:
                best_fold_mae = v_mae
                no_improvement = 0
            else:
                no_improvement += 1
                
            if no_improvement >= patience:
                break
                
        fold_maes.append(best_fold_mae)
        
        # Reportar el MAE acumulado promedio a Optuna después de este fold
        current_avg_mae = float(np.mean(fold_maes))
        trial.report(current_avg_mae, step=fold_idx)
        
        # Verificar si debemos podar (pruning) este trial para ahorrar tiempo si los primeros folds ya son muy malos
        if trial.should_prune():
            raise optuna.TrialPruned()
        
    # Retornar el MAE promedio final de validación a través de todos los folds
    mean_cv_mae = float(np.mean(fold_maes))
    return mean_cv_mae

### 2. Creación e Inicio del Estudio

In [3]:
# Definir nombre del estudio y archivo de base de datos SQLite para persistencia
# Esto evita perder el progreso de 6-7 horas en caso de cortes de energía o caídas de ejecución
study_name = "mtde_net_optimization_v2"
storage_name = "sqlite:///optuna_study.db"

# Creamos un estudio persistente que busca minimizar el MAE CV promedio
# Habilitamos el MedianPruner para podar los trials deficientes a partir del primer fold completado (n_warmup_steps=1)
study = optuna.create_study(
    study_name=study_name,
    storage=storage_name,
    load_if_exists=True,
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=1)
)

study.optimize(objective, n_trials=None, timeout=3600)

print("\n=== MEJORES HIPERPARÁMETROS ENCONTRADOS ===")
print(study.best_params)
print(f"Mejor MAE CV obtenido en validación: {study.best_value:.2f}s")

[I 2026-06-06 16:03:03,093] A new study created in RDB with name: mtde_net_optimization_v2
[I 2026-06-06 16:07:34,125] Trial 0 finished with value: 37.18541558583578 and parameters: {'lr': 0.0005523866096130063, 'weight_decay': 0.0005899092241710652, 'batch_size': 32, 'dropout': 0.28498166581917483}. Best is trial 0 with value: 37.18541558583578.
[I 2026-06-06 16:13:20,436] Trial 1 finished with value: 55.910746256510414 and parameters: {'lr': 0.00010309900897925423, 'weight_decay': 0.0021173381474148224, 'batch_size': 32, 'dropout': 0.14704400373314677}. Best is trial 0 with value: 37.18541558583578.
[I 2026-06-06 16:15:12,371] Trial 2 finished with value: 97.21761067708333 and parameters: {'lr': 6.963471857646145e-05, 'weight_decay': 0.00010865834068412975, 'batch_size': 64, 'dropout': 0.13148255212202073}. Best is trial 0 with value: 37.18541558583578.
[I 2026-06-06 16:17:07,797] Trial 3 finished with value: 98.66300710042317 and parameters: {'lr': 5.8997323833339536e-05, 'weight_de


=== MEJORES HIPERPARÁMETROS ENCONTRADOS ===
{'lr': 0.0006913203372081387, 'weight_decay': 0.008320769245698065, 'batch_size': 16, 'dropout': 0.31580871470081956}
Mejor MAE CV obtenido en validación: 35.06s
